# Chapter 7.1: CUDA Programming Primer for LLM Serving

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand the CUDA programming model** — grids, blocks, threads, and their hierarchical organization
2. **Master thread indexing** — compute global thread IDs for multi-dimensional kernels
3. **Use shared memory** effectively for data reuse and inter-thread communication
4. **Apply warp-level primitives** — shuffle, reduce, and ballot operations
5. **Work with atomic operations** for safe concurrent updates
6. **Manage streams and events** for overlapping computation and data transfer
7. **Write practical kernels** — vector addition, matrix multiplication (naive → tiled → shared memory)
8. **Build and run CUDA kernels from Python** using ctypes, PyCUDA, and CuPy

## Prerequisites

- Basic Python programming
- Understanding of linear algebra (matrix multiplication)
- NVIDIA GPU with CUDA toolkit installed
- `torch`, `cupy`, and `pycuda` Python packages

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part7/chapter_7.1_cuda_primer.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part7/chapter_7.1_cuda_primer.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. The CUDA Programming Model

### 1.1 Why CUDA for LLM Serving?

Large Language Model (LLM) serving is fundamentally a GPU-bound workload. The key operations — matrix multiplications, attention computations, normalization layers — all benefit enormously from the massive parallelism GPUs offer.

While frameworks like PyTorch abstract GPU computation, understanding CUDA is essential for:

- **Custom kernel development**: PagedAttention, fused normalization, quantized matmul
- **Performance optimization**: Understanding why a kernel is slow requires CUDA knowledge
- **Debugging memory issues**: Shared memory bank conflicts, coalescing problems
- **Reading vLLM/SGLang source code**: Many critical paths use custom CUDA kernels

### 1.2 The Execution Hierarchy

CUDA organizes computation in a three-level hierarchy:

```
Grid (entire problem)
 └── Block (cooperative thread group)
      └── Thread (single execution unit)
```

**Key relationships:**

| Level | Max Dimensions | Shared Resources | Synchronization |
|-------|---------------|------------------|-----------------|
| Grid | 3D (2³¹-1 × 65535 × 65535) | Global memory | Kernel boundary |
| Block | 3D (1024 × 1024 × 64, max 1024 total) | Shared memory, L1 cache | `__syncthreads()` |
| Thread | N/A | Registers | N/A |

A **warp** is a group of 32 consecutive threads within a block that execute in lockstep (SIMT). This is crucial for understanding performance.

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, ax = plt.subplots(1, 1, figsize=(12, 8))ax.set_xlim(0, 12)ax.set_ylim(0, 10)ax.axis('off')fig.patch.set_facecolor('white')ax.text(6, 9.5, 'CUDA Execution Model: Grid > Block > Warp > Thread',        fontsize=14, fontweight='bold', ha='center', color=DARK_GRAY)# Grid (outermost)grid_box = mpatches.FancyBboxPatch((0.5, 0.5), 11, 8.3, boxstyle="round,pad=0.2",                                    facecolor=BLUE, alpha=0.15, edgecolor=BLUE, lw=3)ax.add_patch(grid_box)ax.text(1.2, 8.3, 'GRID', fontsize=13, fontweight='bold', color=BLUE)ax.text(1.2, 7.8, '(entire GPU)', fontsize=9, color=BLUE)# Blocksfor bi, (bx, by) in enumerate([(1.5, 1), (6.5, 1)]):    block_box = mpatches.FancyBboxPatch((bx, by), 4.5, 6.2, boxstyle="round,pad=0.15",                                        facecolor=GREEN, alpha=0.15, edgecolor=GREEN, lw=2)    ax.add_patch(block_box)    ax.text(bx + 0.3, 6.8, f'BLOCK {bi}', fontsize=11, fontweight='bold', color=GREEN)    ax.text(bx + 0.3, 6.3, '(runs on 1 SM)', fontsize=8, color=GREEN)    # Warps within each block    for wi, wy in enumerate([4.8, 2.8]):        warp_box = mpatches.FancyBboxPatch((bx+0.3, wy), 3.9, 1.6, boxstyle="round,pad=0.1",                                           facecolor=ORANGE, alpha=0.15, edgecolor=ORANGE, lw=1.5)        ax.add_patch(warp_box)        ax.text(bx + 0.6, wy + 1.2, f'Warp {wi}', fontsize=9, fontweight='bold', color=ORANGE)        ax.text(bx + 0.6, wy + 0.8, '(32 threads, lockstep)', fontsize=7, color=ORANGE)        # Threads within each warp        for ti in range(8):            tx = bx + 0.5 + ti * 0.45            ty = wy + 0.15            thread_box = plt.Rectangle((tx, ty), 0.35, 0.4, facecolor=RED, alpha=0.7,                                       edgecolor='white', lw=0.5)            ax.add_patch(thread_box)            if ti < 4:                ax.text(tx + 0.175, ty + 0.2, f'T{ti}', ha='center', va='center',                        fontsize=6, color='white', fontweight='bold')        ax.text(bx + 4, ty + 0.2, '...', fontsize=8, color=RED, va='center')# Legendlegend_items = [    ("Grid (entire GPU)", BLUE), ("Block (1 SM)", GREEN),    ("Warp (32 threads)", ORANGE), ("Thread (1 core)", RED)]for i, (label, color) in enumerate(legend_items):    ax.add_patch(plt.Rectangle((0.8 + i*2.8, -0.1), 0.25, 0.25, facecolor=color, alpha=0.7))    ax.text(1.15 + i*2.8, 0.02, label, fontsize=8, va='center', color=DARK_GRAY)plt.tight_layout()plt.savefig("cuda_hierarchy.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

### 1.3 Hardware Mapping

Understanding how the software hierarchy maps to hardware is critical:

```
Software          Hardware
--------          --------
Grid        →     GPU (entire device)
Block       →     Streaming Multiprocessor (SM)
Warp        →     Warp Scheduler (each SM has 4)
Thread      →     CUDA Core (INT/FP unit)
```

**NVIDIA GPU Architecture (A100 example):**

| Component | Count | Capability |
|-----------|-------|------------|
| SMs | 108 | Each runs multiple blocks concurrently |
| CUDA Cores/SM | 64 FP32 | 6912 total FP32 cores |
| Tensor Cores/SM | 4 (3rd gen) | 432 total |
| Shared Mem/SM | Up to 164 KB | Configurable with L1 |
| Registers/SM | 65536 × 32-bit | Partitioned across threads |
| Warp Schedulers/SM | 4 | Issue instructions each cycle |

In [ ]:
# Let's query our GPU properties
import torch

if torch.cuda.is_available():
    device = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device)
    
    print(f"GPU: {props.name}")
    print(f"Compute Capability: {props.major}.{props.minor}")
    print(f"Total Global Memory: {props.total_mem / 1e9:.2f} GB")
    print(f"Multiprocessors (SMs): {props.multi_processor_count}")
    print(f"Max Threads per Block: {props.max_threads_per_block}")
    print(f"Max Block Dimensions: ({props.max_block_dim[0]}, {props.max_block_dim[1]}, {props.max_block_dim[2]})")
    print(f"Max Grid Dimensions: ({props.max_grid_dim[0]}, {props.max_grid_dim[1]}, {props.max_grid_dim[2]})")
    print(f"Warp Size: {props.warp_size}")
    print(f"Max Shared Memory per Block: {props.max_shared_memory_per_block / 1024:.1f} KB")
    print(f"Max Registers per Block: {props.regs_per_block}")
else:
    print("No CUDA GPU available. This notebook requires a GPU.")

## 2. Thread Indexing and Memory Addressing

### 2.1 Computing Global Thread IDs

Every CUDA kernel needs to determine which data element each thread processes. The built-in variables are:

- `threadIdx.{x,y,z}` — thread index within its block
- `blockIdx.{x,y,z}` — block index within the grid
- `blockDim.{x,y,z}` — number of threads per block
- `gridDim.{x,y,z}` — number of blocks in the grid

**1D Global Thread ID:**
```c
int tid = blockIdx.x * blockDim.x + threadIdx.x;
```

**2D Global Thread ID (for matrices):**
```c
int row = blockIdx.y * blockDim.y + threadIdx.y;
int col = blockIdx.x * blockDim.x + threadIdx.x;
int idx = row * width + col;  // row-major linear index
```

### 2.2 Boundary Checking

The grid often doesn't align perfectly with the data size. Always guard against out-of-bounds access:

```c
__global__ void kernel(float* data, int N) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= N) return;  // Critical: prevent out-of-bounds!
    data[tid] = data[tid] * 2.0f;
}
```

In [ ]:
# Visualize thread indexing with a concrete example
import numpy as np

def simulate_cuda_indexing(grid_dim, block_dim, data_size):
    """
    Simulate CUDA thread indexing in Python to build intuition.
    
    Args:
        grid_dim: Number of blocks in the grid
        block_dim: Number of threads per block
        data_size: Total number of data elements
    """
    print(f"Grid: {grid_dim} blocks × {block_dim} threads/block = {grid_dim * block_dim} total threads")
    print(f"Data size: {data_size} elements")
    print(f"Excess threads: {grid_dim * block_dim - data_size}")
    print()
    
    for block_idx in range(grid_dim):
        for thread_idx in range(block_dim):
            global_tid = block_idx * block_dim + thread_idx
            status = "ACTIVE" if global_tid < data_size else "IDLE (boundary)"
            if block_idx < 2 or block_idx == grid_dim - 1:  # Show first 2 and last block
                print(f"  Block {block_idx}, Thread {thread_idx} → Global TID {global_tid} [{status}]")
        if block_idx == 1 and grid_dim > 3:
            print(f"  ... ({grid_dim - 3} blocks omitted) ...")

# Example: 10 data elements, 4 threads per block
data_size = 10
block_dim = 4
grid_dim = (data_size + block_dim - 1) // block_dim  # Ceiling division

simulate_cuda_indexing(grid_dim, block_dim, data_size)

## 3. CUDA Memory Hierarchy

### 3.1 Memory Types

Understanding the memory hierarchy is the single most important factor in CUDA performance:

```
                  Bandwidth    Latency    Scope         Size
                  ---------    -------    -----         ----
Registers         ~20 TB/s     0 cycles   Thread        ~256 KB/SM (partitioned)
Shared Memory     ~19 TB/s     ~20 cycles Block         Up to 228 KB/SM (H100)
L1 Cache          ~19 TB/s     ~34 cycles Block (auto)  Configurable with Shared
L2 Cache          ~6 TB/s      ~200 cycles Device       40 MB (A100)
Global (HBM)      ~2 TB/s      ~400 cycles Device       80 GB (A100)
```

### 3.2 Memory Coalescing

When threads in a warp access consecutive memory addresses, the hardware **coalesces** these into a single memory transaction. This is critical for performance:

```c
// GOOD: Coalesced access — threads access consecutive addresses
float val = data[threadIdx.x];  // Thread 0→data[0], Thread 1→data[1], ...

// BAD: Strided access — threads access non-consecutive addresses
float val = data[threadIdx.x * stride];  // Scattered memory transactions
```

### 3.3 Shared Memory and Bank Conflicts

Shared memory is divided into 32 **banks** (one per warp thread). Consecutive 4-byte words are assigned to consecutive banks. A **bank conflict** occurs when multiple threads in a warp access different addresses in the same bank.

```c
// No bank conflict: consecutive access
__shared__ float smem[256];
smem[threadIdx.x] = ...;  // Thread k accesses bank k%32

// Bank conflict: stride-2 access
smem[threadIdx.x * 2] = ...;  // Threads 0,16 both access bank 0!
```

In [ ]:
# Demonstrate memory coalescing impact using PyTorch
import torch
import time

if torch.cuda.is_available():
    N = 1024 * 1024 * 64  # 64M elements
    
    # Create a large tensor
    data = torch.randn(N, device='cuda')
    
    # Coalesced access pattern: sequential read
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(100):
        result = data.sum()  # Reads sequentially → coalesced
    torch.cuda.synchronize()
    coalesced_time = time.perf_counter() - start
    
    # Non-coalesced pattern: strided access
    data_2d = data.view(1024, -1)  # Reshape to 2D
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(100):
        result = data_2d.sum(dim=0)  # Column-wise sum → strided access on row-major
    torch.cuda.synchronize()
    strided_time = time.perf_counter() - start
    
    print(f"Coalesced (row sum):   {coalesced_time*1000:.2f} ms")
    print(f"Strided (column sum):  {strided_time*1000:.2f} ms")
    print(f"Ratio: {strided_time/coalesced_time:.2f}x slower")
else:
    print("GPU not available")

## 4. Shared Memory Usage Patterns

### 4.1 When to Use Shared Memory

Shared memory is essential when:
1. **Multiple threads read the same data** — load once from global, reuse from shared
2. **Inter-thread communication** — threads produce data consumed by others in the block
3. **Data reordering** — convert non-coalesced global accesses to coalesced

### 4.2 Shared Memory Tiling Pattern

The most common pattern in LLM kernels is **tiling**: load a tile of data into shared memory, compute on it, then load the next tile.

```c
__global__ void tiled_kernel(float* A, float* B, float* C, int N) {
    __shared__ float tile_A[TILE_SIZE][TILE_SIZE];
    __shared__ float tile_B[TILE_SIZE][TILE_SIZE];
    
    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;
    float sum = 0.0f;
    
    // Process tiles along the K dimension
    for (int t = 0; t < N / TILE_SIZE; t++) {
        // Step 1: Cooperatively load tiles into shared memory
        tile_A[threadIdx.y][threadIdx.x] = A[row * N + t * TILE_SIZE + threadIdx.x];
        tile_B[threadIdx.y][threadIdx.x] = B[(t * TILE_SIZE + threadIdx.y) * N + col];
        __syncthreads();  // Ensure all data is loaded
        
        // Step 2: Compute on the tile
        for (int k = 0; k < TILE_SIZE; k++) {
            sum += tile_A[threadIdx.y][k] * tile_B[k][threadIdx.x];
        }
        __syncthreads();  // Ensure computation is done before next load
    }
    
    C[row * N + col] = sum;
}
```

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, ax = plt.subplots(1, 1, figsize=(14, 7))ax.set_xlim(0, 14)ax.set_ylim(0, 8)ax.axis('off')fig.patch.set_facecolor('white')ax.text(7, 7.5, 'Shared Memory Tiling for Matrix Multiplication',        fontsize=14, fontweight='bold', ha='center', color=DARK_GRAY)# Matrix Aax.text(2, 6.8, 'Matrix A (M x K)', fontsize=10, fontweight='bold', ha='center', color=BLUE)for i in range(4):    for j in range(4):        color = BLUE if (i < 2 and j < 2) else LIGHT_BLUE        alpha = 0.9 if (i < 2 and j < 2) else 0.3        rect = plt.Rectangle((0.5 + j*0.8, 5.8 - i*0.8), 0.7, 0.7,                              facecolor=color, alpha=alpha, edgecolor='white', lw=1)        ax.add_patch(rect)if True:    ax.text(1.3, 4.0, 'Tile loaded to\nshared memory', fontsize=8, color=BLUE,            fontweight='bold', ha='center',            bbox=dict(boxstyle='round,pad=0.2', facecolor=LIGHT_BLUE, alpha=0.3))# Matrix Bax.text(7, 6.8, 'Matrix B (K x N)', fontsize=10, fontweight='bold', ha='center', color=GREEN)for i in range(4):    for j in range(4):        color = GREEN if (i < 2 and j < 2) else LIGHT_GREEN        alpha = 0.9 if (i < 2 and j < 2) else 0.3        rect = plt.Rectangle((5.5 + j*0.8, 5.8 - i*0.8), 0.7, 0.7,                              facecolor=color, alpha=alpha, edgecolor='white', lw=1)        ax.add_patch(rect)# Matrix C (result)ax.text(12, 6.8, 'Matrix C (M x N)', fontsize=10, fontweight='bold', ha='center', color=RED)for i in range(4):    for j in range(4):        color = RED if (i < 2 and j < 2) else LIGHT_RED        alpha = 0.9 if (i < 2 and j < 2) else 0.15        rect = plt.Rectangle((10.5 + j*0.8, 5.8 - i*0.8), 0.7, 0.7,                              facecolor=color, alpha=alpha, edgecolor='white', lw=1)        ax.add_patch(rect)# Arrowsax.annotate('', xy=(5.2, 5.5), xytext=(4, 5.5),            arrowprops=dict(arrowstyle='->', color=DARK_GRAY, lw=2))ax.annotate('', xy=(10.2, 5.5), xytext=(9, 5.5),            arrowprops=dict(arrowstyle='->', color=DARK_GRAY, lw=2))ax.text(4.6, 5.8, '@', fontsize=14, fontweight='bold', color=DARK_GRAY, ha='center')ax.text(9.6, 5.8, '=', fontsize=14, fontweight='bold', color=DARK_GRAY, ha='center')# Explanationsteps = [    "1. Load tile of A and B into shared memory (fast, on-chip)",    "2. Compute partial product using tile data (reused many times)",    "3. Move to next tile along K dimension, accumulate result",    "4. Result: each element of A/B loaded once from HBM, reused TILE_SIZE times"]for i, step in enumerate(steps):    ax.text(7, 2.8 - i*0.5, step, ha='center', fontsize=9, color=DARK_GRAY)# Memory traffic comparisonax.text(7, 0.5, 'HBM reads reduced by TILE_SIZE (e.g., 16x fewer global memory accesses)',        ha='center', fontsize=10, fontweight='bold', color=GREEN,        bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT_GREEN, alpha=0.3))plt.tight_layout()plt.savefig("shared_memory_tiling.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
# Simulate the tiling pattern in Python to understand data flow
import numpy as np

def simulate_tiled_matmul(N, TILE_SIZE):
    """
    Simulate tiled matrix multiplication to visualize the access pattern.
    
    This mirrors exactly what the CUDA kernel does, but runs on CPU
    to help build intuition about the tiling strategy.
    """
    A = np.random.randn(N, N).astype(np.float32)
    B = np.random.randn(N, N).astype(np.float32)
    C = np.zeros((N, N), dtype=np.float32)
    
    num_tiles = N // TILE_SIZE
    total_global_loads = 0
    
    print(f"Matrix size: {N}×{N}, Tile size: {TILE_SIZE}×{TILE_SIZE}")
    print(f"Number of tiles per dimension: {num_tiles}")
    print(f"Number of thread blocks: {num_tiles * num_tiles}")
    print()
    
    # Simulate each block
    for block_row in range(num_tiles):
        for block_col in range(num_tiles):
            # This block computes C[block_row*T:(block_row+1)*T, block_col*T:(block_col+1)*T]
            accumulator = np.zeros((TILE_SIZE, TILE_SIZE), dtype=np.float32)
            
            for t in range(num_tiles):
                # Load tiles from global memory into shared memory
                tile_A = A[block_row*TILE_SIZE:(block_row+1)*TILE_SIZE,
                          t*TILE_SIZE:(t+1)*TILE_SIZE]
                tile_B = B[t*TILE_SIZE:(t+1)*TILE_SIZE,
                          block_col*TILE_SIZE:(block_col+1)*TILE_SIZE]
                
                total_global_loads += 2 * TILE_SIZE * TILE_SIZE  # Each tile load
                
                # Compute on tiles (this happens in shared memory)
                accumulator += tile_A @ tile_B
            
            # Store result back to global memory
            C[block_row*TILE_SIZE:(block_row+1)*TILE_SIZE,
              block_col*TILE_SIZE:(block_col+1)*TILE_SIZE] = accumulator
    
    # Verify correctness
    C_ref = A @ B
    max_error = np.max(np.abs(C - C_ref))
    
    # Compare with naive approach
    naive_global_loads = N * N * N * 2  # Each C[i,j] reads entire row of A and col of B
    
    print(f"Tiled global loads:  {total_global_loads:,}")
    print(f"Naive global loads:  {naive_global_loads:,}")
    print(f"Reduction factor:    {naive_global_loads / total_global_loads:.1f}x")
    print(f"Max error: {max_error:.2e}")

simulate_tiled_matmul(N=16, TILE_SIZE=4)

## 5. Warp-Level Operations

### 5.1 What is a Warp?

A warp is a group of 32 threads that execute **in lockstep** on the GPU. All threads in a warp execute the same instruction simultaneously (SIMT — Single Instruction, Multiple Threads). This has profound implications:

- **Warp divergence**: When threads in a warp take different branch paths, both paths execute sequentially, reducing efficiency
- **Warp-level primitives**: Threads can directly exchange data within a warp without shared memory

### 5.2 Warp Shuffle Operations

Warp shuffles allow threads to read registers from other threads in the same warp:

```c
// __shfl_sync: Read from a specific lane
float val = __shfl_sync(0xFFFFFFFF, my_val, src_lane);

// __shfl_xor_sync: Read from lane = my_lane XOR mask (butterfly pattern)
float val = __shfl_xor_sync(0xFFFFFFFF, my_val, mask);

// __shfl_down_sync: Read from lane = my_lane + delta
float val = __shfl_down_sync(0xFFFFFFFF, my_val, delta);

// __shfl_up_sync: Read from lane = my_lane - delta
float val = __shfl_up_sync(0xFFFFFFFF, my_val, delta);
```

### 5.3 Warp Reduce Pattern

The most important warp-level pattern for LLM serving is the **warp reduce**, used in softmax, layernorm, and attention:

```c
// Warp-level reduction to find the sum of values across all 32 threads
__device__ float warp_reduce_sum(float val) {
    for (int offset = 16; offset > 0; offset >>= 1) {
        val += __shfl_down_sync(0xFFFFFFFF, val, offset);
    }
    return val;  // Only lane 0 has the final result
}

// Warp-level reduction to find the maximum
__device__ float warp_reduce_max(float val) {
    for (int offset = 16; offset > 0; offset >>= 1) {
        val = fmaxf(val, __shfl_down_sync(0xFFFFFFFF, val, offset));
    }
    return val;  // Only lane 0 has the final result
}
```

This takes only **5 steps** (log2(32) = 5) to reduce 32 values, and uses **no shared memory**!

In [ ]:
# Simulate warp shuffle reduction to visualize the data flow
import numpy as np

def simulate_warp_reduce_sum(values):
    """
    Simulate a warp-level sum reduction using __shfl_down_sync.
    
    This shows exactly how 32 threads cooperate to compute a sum
    in just 5 steps using register-to-register communication.
    """
    assert len(values) == 32, "Warp has exactly 32 lanes"
    warp = values.copy()
    
    print(f"Initial values: {warp[:8]}... (showing first 8 lanes)")
    print(f"Expected sum: {sum(values):.4f}")
    print()
    
    step = 0
    for offset in [16, 8, 4, 2, 1]:
        step += 1
        new_warp = warp.copy()
        active_lanes = []
        for lane in range(32):
            src_lane = lane + offset
            if src_lane < 32:
                new_warp[lane] = warp[lane] + warp[src_lane]
                if lane < 4:  # Show first few lanes
                    active_lanes.append(f"lane[{lane}] += lane[{src_lane}]")
        warp = new_warp
        print(f"Step {step} (offset={offset:2d}): {', '.join(active_lanes)}, ...")
        print(f"  Lane 0 now holds: {warp[0]:.4f}")
    
    print(f"\nFinal result in lane 0: {warp[0]:.4f}")
    print(f"Correct: {abs(warp[0] - sum(values)) < 1e-4}")

np.random.seed(42)
values = np.random.randn(32).astype(np.float32).tolist()
simulate_warp_reduce_sum(values)

## 6. Atomic Operations

### 6.1 Why Atomics?

When multiple threads need to update the same memory location, **race conditions** occur. Atomic operations ensure that read-modify-write sequences are indivisible:

```c
// Without atomic: RACE CONDITION!
output[0] += local_sum;  // Two threads read old value, both add, one write is lost

// With atomic: CORRECT but slower
atomicAdd(&output[0], local_sum);  // Serialized but correct
```

### 6.2 Available Atomic Operations

| Operation | Types | Use Case |
|-----------|-------|----------|
| `atomicAdd` | int, float, double | Reduction, histogram |
| `atomicMax/Min` | int | Finding extremes |
| `atomicCAS` | int, unsigned | Lock-free data structures |
| `atomicExch` | int, float | Swapping values |
| `atomicAnd/Or/Xor` | int | Bitmask operations |

### 6.3 Performance Implications

Atomics are slow because they serialize access. In LLM kernels, we minimize atomics by:
1. **Warp-level reduction first**, then atomic the warp result
2. **Block-level reduction** with shared memory, then atomic the block result
3. This reduces atomics from N threads to N/block_size blocks

In [ ]:
# Demonstrate the hierarchical reduction pattern used in LLM kernels
def simulate_hierarchical_reduction(N, block_size=256):
    """
    Show why hierarchical reduction (warp → block → global) is used
    instead of naive atomicAdd per thread.
    
    In practice:
    - Each thread computes a partial result
    - Warp reduce: 32 threads → 1 value (5 shuffle steps, 0 shared mem)
    - Block reduce: N_warps values → 1 value (shared memory)
    - Global reduce: N_blocks values → 1 value (atomicAdd)
    """
    warp_size = 32
    warps_per_block = block_size // warp_size
    num_blocks = (N + block_size - 1) // block_size
    
    print(f"Problem size: {N:,} elements")
    print(f"Block size: {block_size} threads ({warps_per_block} warps)")
    print(f"Grid size: {num_blocks} blocks")
    print()
    
    # Approach 1: Naive atomicAdd per thread
    naive_atomics = N
    
    # Approach 2: Warp reduce + atomicAdd
    warp_reduce_atomics = N // warp_size
    
    # Approach 3: Warp reduce + block reduce + atomicAdd
    hierarchical_atomics = num_blocks
    
    print(f"{'Approach':<35} {'Atomic Operations':>20} {'Relative':>10}")
    print("-" * 65)
    print(f"{'Naive (1 atomic/thread)':<35} {naive_atomics:>20,} {naive_atomics/naive_atomics:>10.1f}x")
    print(f"{'Warp reduce + atomic':<35} {warp_reduce_atomics:>20,} {warp_reduce_atomics/naive_atomics:>10.4f}x")
    print(f"{'Warp + block reduce + atomic':<35} {hierarchical_atomics:>20,} {hierarchical_atomics/naive_atomics:>10.6f}x")

simulate_hierarchical_reduction(N=1_000_000)

## 7. Streams and Events

### 7.1 CUDA Streams

A **stream** is a sequence of operations that execute in order on the GPU. Operations in different streams can execute concurrently:

```c
cudaStream_t stream1, stream2;
cudaStreamCreate(&stream1);
cudaStreamCreate(&stream2);

// These can overlap: copy in stream1 while computing in stream2
cudaMemcpyAsync(d_A, h_A, size, cudaMemcpyHostToDevice, stream1);
kernel<<<grid, block, 0, stream2>>>(d_B, d_C);
```

### 7.2 In LLM Serving

Streams are critical for LLM serving performance:

- **Overlap KV cache transfer with computation**: While computing attention for one request, copy KV cache for the next
- **Overlap prefill and decode**: Different streams for different batch types
- **Pipeline parallelism**: Each pipeline stage runs in its own stream

### 7.3 CUDA Events

Events are synchronization markers inserted into streams:

```c
cudaEvent_t start, stop;
cudaEventCreate(&start);
cudaEventCreate(&stop);

cudaEventRecord(start, stream);
kernel<<<grid, block, 0, stream>>>(...);
cudaEventRecord(stop, stream);
cudaEventSynchronize(stop);

float ms;
cudaEventElapsedTime(&ms, start, stop);
```

In [ ]:
# Demonstrate CUDA streams in PyTorch
import torch
import time

if torch.cuda.is_available():
    # Create two streams
    stream1 = torch.cuda.Stream()
    stream2 = torch.cuda.Stream()
    
    N = 4096
    A = torch.randn(N, N, device='cuda')
    B = torch.randn(N, N, device='cuda')
    C = torch.randn(N, N, device='cuda')
    D = torch.randn(N, N, device='cuda')
    
    # Sequential execution
    torch.cuda.synchronize()
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    start_event.record()
    result1 = torch.mm(A, B)
    result2 = torch.mm(C, D)
    end_event.record()
    torch.cuda.synchronize()
    sequential_time = start_event.elapsed_time(end_event)
    
    # Concurrent execution with streams
    start_event.record()
    with torch.cuda.stream(stream1):
        result1 = torch.mm(A, B)
    with torch.cuda.stream(stream2):
        result2 = torch.mm(C, D)
    end_event.record()
    torch.cuda.synchronize()
    concurrent_time = start_event.elapsed_time(end_event)
    
    print(f"Sequential: {sequential_time:.2f} ms")
    print(f"Concurrent: {concurrent_time:.2f} ms")
    print(f"Speedup: {sequential_time/concurrent_time:.2f}x")
    print()
    print("Note: Speedup may be limited if the GPU is already fully utilized by a single matmul.")
    print("Streams are most beneficial when individual operations don't saturate the GPU.")
else:
    print("GPU not available")

## 8. Writing CUDA Kernels from Python

### 8.1 Approach 1: CuPy — Easiest

CuPy provides `RawKernel` to compile and run CUDA code directly from Python strings. This is the easiest way to experiment with CUDA kernels.

In [ ]:
# Vector Addition — The "Hello World" of CUDA
try:
    import cupy as cp
    
    # Define the CUDA kernel as a string
    vector_add_kernel = cp.RawKernel(r'''
    extern "C" __global__
    void vector_add(const float* A, const float* B, float* C, int N) {
        // Step 1: Compute global thread ID
        int tid = blockIdx.x * blockDim.x + threadIdx.x;
        
        // Step 2: Boundary check — not all threads have valid work
        if (tid >= N) return;
        
        // Step 3: Perform the addition
        C[tid] = A[tid] + B[tid];
    }
    ''', 'vector_add')
    
    # Prepare data
    N = 1_000_000
    A = cp.random.randn(N, dtype=cp.float32)
    B = cp.random.randn(N, dtype=cp.float32)
    C = cp.zeros(N, dtype=cp.float32)
    
    # Launch configuration
    block_size = 256
    grid_size = (N + block_size - 1) // block_size  # Ceiling division
    
    print(f"N = {N:,}")
    print(f"Block size: {block_size}")
    print(f"Grid size: {grid_size}")
    print(f"Total threads: {grid_size * block_size:,}")
    print(f"Excess threads: {grid_size * block_size - N:,}")
    
    # Launch kernel
    vector_add_kernel((grid_size,), (block_size,), (A, B, C, N))
    
    # Verify
    expected = A + B
    assert cp.allclose(C, expected), "Kernel output doesn't match!"
    print("\nVector addition kernel: PASSED!")
    
    # Benchmark
    import time
    cp.cuda.Stream.null.synchronize()
    start = time.perf_counter()
    for _ in range(1000):
        vector_add_kernel((grid_size,), (block_size,), (A, B, C, N))
    cp.cuda.Stream.null.synchronize()
    elapsed = time.perf_counter() - start
    
    bandwidth = 3 * N * 4 / (elapsed / 1000) / 1e9  # 3 arrays × N elements × 4 bytes
    print(f"Time per kernel: {elapsed/1000*1e6:.1f} µs")
    print(f"Effective bandwidth: {bandwidth:.1f} GB/s")
    
except ImportError:
    print("CuPy not installed. Install with: pip install cupy-cuda12x")
    print("Showing the kernel code for reference.")

In [ ]:
# Matrix Multiplication — Naive Implementation
try:
    import cupy as cp
    
    naive_matmul_kernel = cp.RawKernel(r'''
    extern "C" __global__
    void naive_matmul(const float* A, const float* B, float* C, int M, int N, int K) {
        // A is M×K, B is K×N, C is M×N
        // Each thread computes one element of C
        
        int row = blockIdx.y * blockDim.y + threadIdx.y;  // Row of C
        int col = blockIdx.x * blockDim.x + threadIdx.x;  // Column of C
        
        if (row >= M || col >= N) return;
        
        float sum = 0.0f;
        for (int k = 0; k < K; k++) {
            // A[row, k] * B[k, col]
            // Row-major: A[row][k] = A[row * K + k]
            //            B[k][col] = B[k * N + col]
            sum += A[row * K + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
    ''', 'naive_matmul')
    
    # Test
    M, N, K = 512, 512, 512
    A = cp.random.randn(M, K, dtype=cp.float32)
    B = cp.random.randn(K, N, dtype=cp.float32)
    C = cp.zeros((M, N), dtype=cp.float32)
    
    block = (16, 16)  # 16×16 = 256 threads per block
    grid = ((N + 15) // 16, (M + 15) // 16)
    
    naive_matmul_kernel(grid, block, (A, B, C, M, N, K))
    
    # Verify against cupy matmul
    expected = A @ B
    max_error = float(cp.max(cp.abs(C - expected)))
    print(f"Naive matmul ({M}×{K} @ {K}×{N}): max error = {max_error:.6f}")
    assert max_error < 0.01, f"Error too large: {max_error}"
    print("PASSED!")
    
except ImportError:
    print("CuPy not installed.")

In [ ]:
# Matrix Multiplication — Tiled with Shared Memory
try:
    import cupy as cp
    
    TILE_SIZE = 16
    
    tiled_matmul_kernel = cp.RawKernel(r'''
    #define TILE_SIZE 16
    
    extern "C" __global__
    void tiled_matmul(const float* A, const float* B, float* C, int M, int N, int K) {
        // Shared memory tiles for A and B
        __shared__ float tile_A[TILE_SIZE][TILE_SIZE];
        __shared__ float tile_B[TILE_SIZE][TILE_SIZE];
        
        int row = blockIdx.y * TILE_SIZE + threadIdx.y;
        int col = blockIdx.x * TILE_SIZE + threadIdx.x;
        
        float sum = 0.0f;
        
        // Iterate over tiles along K dimension
        int num_tiles = (K + TILE_SIZE - 1) / TILE_SIZE;
        
        for (int t = 0; t < num_tiles; t++) {
            // Load tile of A into shared memory
            int a_col = t * TILE_SIZE + threadIdx.x;
            if (row < M && a_col < K)
                tile_A[threadIdx.y][threadIdx.x] = A[row * K + a_col];
            else
                tile_A[threadIdx.y][threadIdx.x] = 0.0f;
            
            // Load tile of B into shared memory
            int b_row = t * TILE_SIZE + threadIdx.y;
            if (b_row < K && col < N)
                tile_B[threadIdx.y][threadIdx.x] = B[b_row * N + col];
            else
                tile_B[threadIdx.y][threadIdx.x] = 0.0f;
            
            // Synchronize to ensure tiles are fully loaded
            __syncthreads();
            
            // Compute partial sum from this tile
            for (int k = 0; k < TILE_SIZE; k++) {
                sum += tile_A[threadIdx.y][k] * tile_B[k][threadIdx.x];
            }
            
            // Synchronize before loading next tile
            __syncthreads();
        }
        
        // Write result
        if (row < M && col < N)
            C[row * N + col] = sum;
    }
    ''', 'tiled_matmul')
    
    # Test
    M, N, K = 512, 512, 512
    A = cp.random.randn(M, K, dtype=cp.float32)
    B = cp.random.randn(K, N, dtype=cp.float32)
    C_tiled = cp.zeros((M, N), dtype=cp.float32)
    
    block = (TILE_SIZE, TILE_SIZE)
    grid = ((N + TILE_SIZE - 1) // TILE_SIZE, (M + TILE_SIZE - 1) // TILE_SIZE)
    
    tiled_matmul_kernel(grid, block, (A, B, C_tiled, M, N, K))
    
    expected = A @ B
    max_error = float(cp.max(cp.abs(C_tiled - expected)))
    print(f"Tiled matmul ({M}×{K} @ {K}×{N}): max error = {max_error:.6f}")
    print("PASSED!" if max_error < 0.01 else "FAILED!")
    
except ImportError:
    print("CuPy not installed.")

In [ ]:
# Benchmark: Naive vs Tiled vs cuBLAS
try:
    import cupy as cp
    import time
    
    sizes = [128, 256, 512, 1024]
    results = []
    
    for size in sizes:
        M = N = K = size
        A = cp.random.randn(M, K, dtype=cp.float32)
        B = cp.random.randn(K, N, dtype=cp.float32)
        
        # Naive
        C_naive = cp.zeros((M, N), dtype=cp.float32)
        block = (16, 16)
        grid = ((N + 15) // 16, (M + 15) // 16)
        
        cp.cuda.Stream.null.synchronize()
        start = time.perf_counter()
        for _ in range(10):
            naive_matmul_kernel(grid, block, (A, B, C_naive, M, N, K))
        cp.cuda.Stream.null.synchronize()
        naive_time = (time.perf_counter() - start) / 10 * 1000
        
        # Tiled
        C_tiled = cp.zeros((M, N), dtype=cp.float32)
        
        cp.cuda.Stream.null.synchronize()
        start = time.perf_counter()
        for _ in range(10):
            tiled_matmul_kernel(grid, block, (A, B, C_tiled, M, N, K))
        cp.cuda.Stream.null.synchronize()
        tiled_time = (time.perf_counter() - start) / 10 * 1000
        
        # cuBLAS (via CuPy)
        cp.cuda.Stream.null.synchronize()
        start = time.perf_counter()
        for _ in range(10):
            C_cublas = A @ B
        cp.cuda.Stream.null.synchronize()
        cublas_time = (time.perf_counter() - start) / 10 * 1000
        
        # GFLOPS
        flops = 2 * M * N * K  # Multiply-add = 2 ops
        results.append({
            'size': size,
            'naive_ms': naive_time,
            'tiled_ms': tiled_time,
            'cublas_ms': cublas_time,
            'naive_gflops': flops / (naive_time / 1000) / 1e9,
            'tiled_gflops': flops / (tiled_time / 1000) / 1e9,
            'cublas_gflops': flops / (cublas_time / 1000) / 1e9,
        })
    
    print(f"{'Size':>6} | {'Naive (ms)':>10} {'GFLOPS':>8} | {'Tiled (ms)':>10} {'GFLOPS':>8} | {'cuBLAS (ms)':>11} {'GFLOPS':>8} | {'Tiled Speedup':>14}")
    print("-" * 100)
    for r in results:
        print(f"{r['size']:>6} | {r['naive_ms']:>10.3f} {r['naive_gflops']:>8.1f} | {r['tiled_ms']:>10.3f} {r['tiled_gflops']:>8.1f} | {r['cublas_ms']:>11.3f} {r['cublas_gflops']:>8.1f} | {r['naive_ms']/r['tiled_ms']:>13.2f}x")

except ImportError:
    print("CuPy not installed.")
except Exception as e:
    print(f"Error: {e}")

## 9. Practical Tips for LLM Kernel Development

### 9.1 Common Patterns in LLM Kernels

| Pattern | Where Used | Key Technique |
|---------|-----------|---------------|
| **Reduction** | Softmax, LayerNorm, RMSNorm | Warp shuffle → shared mem → atomic |
| **Elementwise** | Activation (SiLU, GELU), RoPE | Simple 1D mapping, high bandwidth |
| **GEMM** | Linear layers, attention QKV | Tiling, tensor cores, CUTLASS |
| **Gather/Scatter** | PagedAttention, KV cache | Block table lookup, indirect addressing |
| **Fused ops** | RMSNorm+residual, SiLU*gate | Combine multiple ops to save memory traffic |

### 9.2 Performance Checklist

1. **Memory coalescing**: Ensure warp threads access consecutive addresses
2. **Occupancy**: Use enough threads/block to hide memory latency (usually 128-256)
3. **Shared memory**: Use for data reuse, but watch bank conflicts
4. **Register pressure**: Too many variables → register spilling → slow
5. **Warp divergence**: Minimize branch conditions within a warp
6. **Instruction-level parallelism**: Interleave independent operations

### 9.3 Debugging Tips

- Always verify kernel output against a reference (NumPy/PyTorch)
- Start with small sizes where you can manually verify
- Use `printf` inside kernels (limited, but useful for debugging)
- `cuda-memcheck` catches out-of-bounds access
- Profile with Nsight Compute before optimizing (Chapter 7.8)

In [ ]:
# Approach 2: Using torch.utils.cpp_extension for inline CUDA
# This is how vLLM integrates custom kernels

cuda_source = '''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void silu_mul_kernel(
    const float* __restrict__ input,
    const float* __restrict__ gate,
    float* __restrict__ output,
    int N
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= N) return;
    
    // SiLU(x) = x * sigmoid(x)
    float x = input[idx];
    float g = gate[idx];
    float silu = x / (1.0f + expf(-x));
    output[idx] = silu * g;
}

torch::Tensor silu_mul_cuda(torch::Tensor input, torch::Tensor gate) {
    auto N = input.numel();
    auto output = torch::empty_like(input);
    
    int block_size = 256;
    int grid_size = (N + block_size - 1) / block_size;
    
    silu_mul_kernel<<<grid_size, block_size>>>(
        input.data_ptr<float>(),
        gate.data_ptr<float>(),
        output.data_ptr<float>(),
        N
    );
    
    return output;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("silu_mul", &silu_mul_cuda, "Fused SiLU-Mul CUDA kernel");
}
'''

print("This CUDA kernel implements fused SiLU activation with gating.")
print("In LLMs (e.g., LLaMA), this replaces two separate operations:")
print("  1. SiLU(x) = x * sigmoid(x)   — activation")
print("  2. SiLU(x) * gate             — gated output")
print()
print("Fusing saves one round-trip to global memory (~2 TB/s on A100).")
print("For a hidden size of 8192 and batch of 256:")
print(f"  Memory saved: {2 * 8192 * 256 * 4 / 1e6:.1f} MB per layer")

In [ ]:
# Compile and run the kernel (requires CUDA toolkit)
import torch

try:
    from torch.utils.cpp_extension import load_inline
    
    # Note: This requires nvcc to be installed
    silu_mul_module = load_inline(
        name='silu_mul',
        cpp_sources='',
        cuda_sources=[cuda_source],
        functions=['silu_mul'],
        verbose=False
    )
    
    # Test
    x = torch.randn(1024, device='cuda')
    gate = torch.randn(1024, device='cuda')
    
    # Custom kernel
    result_custom = silu_mul_module.silu_mul(x, gate)
    
    # Reference (PyTorch)
    result_ref = torch.nn.functional.silu(x) * gate
    
    max_error = (result_custom - result_ref).abs().max().item()
    print(f"Max error: {max_error:.8f}")
    print(f"PASSED: {max_error < 1e-5}")
    
    # Benchmark
    x = torch.randn(8192 * 256, device='cuda')
    gate = torch.randn(8192 * 256, device='cuda')
    
    # Warmup
    for _ in range(10):
        _ = silu_mul_module.silu_mul(x, gate)
        _ = torch.nn.functional.silu(x) * gate
    
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    # Custom kernel
    start.record()
    for _ in range(100):
        _ = silu_mul_module.silu_mul(x, gate)
    end.record()
    torch.cuda.synchronize()
    custom_time = start.elapsed_time(end) / 100
    
    # PyTorch (unfused)
    start.record()
    for _ in range(100):
        _ = torch.nn.functional.silu(x) * gate
    end.record()
    torch.cuda.synchronize()
    pytorch_time = start.elapsed_time(end) / 100
    
    print(f"\nCustom fused kernel: {custom_time:.3f} ms")
    print(f"PyTorch (unfused):   {pytorch_time:.3f} ms")
    print(f"Speedup: {pytorch_time/custom_time:.2f}x")
    
except Exception as e:
    print(f"Could not compile CUDA kernel: {e}")
    print("This requires nvcc (CUDA toolkit) to be installed.")

## 10. Reduction Kernel — A Complete Example

Reduction is one of the most important primitives in LLM kernels. It's used in:
- **Softmax**: Find max, compute sum of exponentials
- **LayerNorm / RMSNorm**: Compute mean and variance
- **Loss computation**: Sum over sequence length

Let's build a complete reduction kernel, step by step:

### Level 1: Naive (shared memory only)
### Level 2: Warp shuffle + shared memory
### Level 3: Multi-block reduction

In [ ]:
try:
    import cupy as cp
    
    # Level 1: Shared memory reduction
    reduction_v1 = cp.RawKernel(r'''
    extern "C" __global__
    void reduce_sum_v1(const float* input, float* output, int N) {
        // Shared memory for this block's partial sums
        extern __shared__ float sdata[];
        
        int tid = threadIdx.x;
        int gid = blockIdx.x * blockDim.x + threadIdx.x;
        
        // Load from global memory into shared memory
        sdata[tid] = (gid < N) ? input[gid] : 0.0f;
        __syncthreads();
        
        // Tree reduction in shared memory
        // Each step halves the active threads
        for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
            if (tid < stride) {
                sdata[tid] += sdata[tid + stride];
            }
            __syncthreads();
        }
        
        // Thread 0 writes the block result
        if (tid == 0) {
            atomicAdd(output, sdata[0]);
        }
    }
    ''', 'reduce_sum_v1')
    
    # Level 2: Warp shuffle + shared memory (optimized)
    reduction_v2 = cp.RawKernel(r'''
    extern "C" __global__
    void reduce_sum_v2(const float* input, float* output, int N) {
        extern __shared__ float sdata[];
        
        int tid = threadIdx.x;
        int gid = blockIdx.x * blockDim.x + threadIdx.x;
        
        float val = (gid < N) ? input[gid] : 0.0f;
        
        // Step 1: Warp-level reduction using shuffle
        // Each warp reduces 32 values to 1 without shared memory
        for (int offset = 16; offset > 0; offset >>= 1) {
            val += __shfl_down_sync(0xFFFFFFFF, val, offset);
        }
        
        // Step 2: Store warp results in shared memory
        int warp_id = tid / 32;
        int lane_id = tid % 32;
        if (lane_id == 0) {
            sdata[warp_id] = val;
        }
        __syncthreads();
        
        // Step 3: Final reduction of warp results (first warp only)
        int num_warps = blockDim.x / 32;
        if (warp_id == 0) {
            val = (lane_id < num_warps) ? sdata[lane_id] : 0.0f;
            for (int offset = 16; offset > 0; offset >>= 1) {
                val += __shfl_down_sync(0xFFFFFFFF, val, offset);
            }
        }
        
        if (tid == 0) {
            atomicAdd(output, val);
        }
    }
    ''', 'reduce_sum_v2')
    
    # Test both versions
    N = 1_000_000
    data = cp.random.randn(N, dtype=cp.float32)
    expected = float(data.sum())
    
    block_size = 256
    grid_size = (N + block_size - 1) // block_size
    shared_mem = block_size * 4  # float32 = 4 bytes
    
    # V1: Shared memory
    output_v1 = cp.zeros(1, dtype=cp.float32)
    reduction_v1((grid_size,), (block_size,), (data, output_v1, N), shared_mem=shared_mem)
    result_v1 = float(output_v1[0])
    
    # V2: Warp shuffle
    warp_shared_mem = (block_size // 32) * 4  # Only need space for warp results
    output_v2 = cp.zeros(1, dtype=cp.float32)
    reduction_v2((grid_size,), (block_size,), (data, output_v2, N), shared_mem=warp_shared_mem)
    result_v2 = float(output_v2[0])
    
    print(f"Expected:  {expected:.4f}")
    print(f"V1 (smem): {result_v1:.4f} (error: {abs(result_v1 - expected):.6f})")
    print(f"V2 (warp): {result_v2:.4f} (error: {abs(result_v2 - expected):.6f})")
    print()
    print(f"Shared memory per block:")
    print(f"  V1: {shared_mem} bytes ({block_size} floats)")
    print(f"  V2: {warp_shared_mem} bytes ({block_size // 32} floats) — {shared_mem / warp_shared_mem:.0f}x less!")
    
    # Benchmark
    import time
    
    for name, kernel, out, smem in [
        ("V1 (shared mem)", reduction_v1, output_v1, shared_mem),
        ("V2 (warp shuffle)", reduction_v2, output_v2, warp_shared_mem),
    ]:
        cp.cuda.Stream.null.synchronize()
        start = time.perf_counter()
        for _ in range(1000):
            out[0] = 0
            kernel((grid_size,), (block_size,), (data, out, N), shared_mem=smem)
        cp.cuda.Stream.null.synchronize()
        elapsed = (time.perf_counter() - start) / 1000 * 1e6
        print(f"{name}: {elapsed:.1f} µs")

except ImportError:
    print("CuPy not installed.")

## 11. Summary and Key Takeaways

### What We Covered

| Topic | Key Insight |
|-------|------------|
| **CUDA hierarchy** | Grid → Block → Warp → Thread; warps execute in lockstep |
| **Thread indexing** | `blockIdx.x * blockDim.x + threadIdx.x`; always check bounds |
| **Memory hierarchy** | Registers > Shared > L1 > L2 > Global; 10x bandwidth gaps |
| **Shared memory** | Tiling pattern: load tile → syncthreads → compute → syncthreads |
| **Warp operations** | Shuffle reduces 32 values in 5 steps without shared memory |
| **Atomics** | Use hierarchical reduction to minimize atomic contention |
| **Streams** | Overlap computation and data transfer for better utilization |

### Relevance to LLM Serving

Every custom kernel in vLLM and SGLang uses these primitives:
- **PagedAttention** (Ch 7.3): Indirect memory access via block table, warp reduction for softmax
- **FlashAttention** (Ch 7.4): Tiling in shared memory, online softmax
- **Fused kernels** (Ch 7.5): Elementwise ops, block-level reduction
- **Quantized matmul** (Ch 7.6): Shared memory tiling with dequantization

### Next Steps

In Chapter 7.2, we'll learn **Triton**, which provides the same GPU programming power but with Python syntax and automatic optimizations — making kernel development significantly more accessible.

## Exercises

### Exercise 1: Implement a Parallel Prefix Sum (Scan)

Implement an inclusive scan kernel: given input `[a, b, c, d]`, produce `[a, a+b, a+b+c, a+b+c+d]`.

Hint: Use the Hillis-Steele or Blelloch algorithm in shared memory.

### Exercise 2: Implement a Histogram Kernel

Given an array of integers in range [0, 255], compute the histogram. Use:
- Shared memory for per-block histograms
- `atomicAdd` for shared memory accumulation
- Final `atomicAdd` for global accumulation

### Exercise 3: Optimize the Matrix Multiplication

Take the tiled matmul kernel and add:
1. **Double buffering**: Load next tile while computing current tile
2. **Thread coarsening**: Each thread computes multiple output elements
3. **Register blocking**: Keep partial sums in registers, not shared memory

### Exercise 4: Implement a Reduction Kernel with Grid-Stride Loop

Instead of launching one thread per element, use a grid-stride loop to handle arbitrary data sizes with a fixed grid:

```c
float sum = 0;
for (int i = gid; i < N; i += gridDim.x * blockDim.x) {
    sum += input[i];
}
// Then reduce 'sum' across the block
```

In [ ]:
# Exercise 1 Skeleton: Prefix Sum
# Fill in the kernel body

prefix_sum_template = r'''
extern "C" __global__
void inclusive_scan(const float* input, float* output, int N) {
    extern __shared__ float sdata[];
    
    int tid = threadIdx.x;
    int gid = blockIdx.x * blockDim.x + threadIdx.x;
    
    // Step 1: Load data into shared memory
    sdata[tid] = (gid < N) ? input[gid] : 0.0f;
    __syncthreads();
    
    // Step 2: Hillis-Steele scan
    // TODO: Implement the scan algorithm
    // Hint: for stride = 1, 2, 4, 8, ...
    //   if (tid >= stride) sdata[tid] += sdata[tid - stride]
    //   syncthreads()
    
    // Step 3: Write result
    if (gid < N) {
        output[gid] = sdata[tid];
    }
}
'''

print("Exercise 1: Implement the Hillis-Steele inclusive scan.")
print("Replace the TODO with the actual scan loop.")
print("Test with input: [1, 2, 3, 4, 5, 6, 7, 8]")
print("Expected output: [1, 3, 6, 10, 15, 21, 28, 36]")

In [ ]:
# Exercise 2 Skeleton: Histogram

histogram_template = r'''
extern "C" __global__
void histogram(const int* input, int* output, int N) {
    // Shared memory histogram for this block
    __shared__ int local_hist[256];
    
    int tid = threadIdx.x;
    int gid = blockIdx.x * blockDim.x + threadIdx.x;
    
    // Step 1: Initialize local histogram to zeros
    // TODO: Each thread should zero out some bins
    // Hint: Use a loop with stride = blockDim.x
    
    __syncthreads();
    
    // Step 2: Accumulate into local histogram
    // TODO: atomicAdd to local_hist
    
    __syncthreads();
    
    // Step 3: Merge local histogram into global histogram
    // TODO: atomicAdd local_hist[tid] into output[tid]
}
'''

print("Exercise 2: Implement the histogram kernel.")
print("Fill in the three TODO sections.")
print("Test with random integers in [0, 255].")